# Mini-Project: MCP + Gemini Agent Integration

**Course:** Developers Institute  **Week 9 - Day 5**  
**Author:** Alex Goldbaum

## Theme — Workspace Assistant
An agentic application where Gemini orchestrates three MCP servers:
1. **filesystem** (third-party, official): read/list/write files inside a working directory.
2. **git** (third-party, official): inspect the git repository (status, log, diff).
3. **workspace_ops** (custom, FastMCP): domain-specific tools — summarize a file,
   count lines per extension, and generate a changelog snippet from a git diff.

The LLM (Gemini 2.0 Flash) decides which tool to call on each step using a LangGraph
ReAct agent — there is no hard-coded workflow. The notebook ends with three example
tasks that exercise composition across all three servers.


## 1. Install dependencies

Run this cell first. It installs LangChain + LangGraph, the Gemini integration,
the MCP adapter for LangChain, and FastMCP for the custom server.


In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "nest_asyncio"


## 2. Set the Google API key

Generate a key at https://aistudio.google.com/apikey and paste it below.
On Colab you can also store it as a secret named `GOOGLE_API_KEY` via the key icon
in the left sidebar, then this cell will pick it up automatically.


In [ ]:
import os

if not os.environ.get('GOOGLE_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    except Exception:
        import getpass
        os.environ['GOOGLE_API_KEY'] = getpass.getpass('Enter your GOOGLE_API_KEY: ')

assert os.environ.get('GOOGLE_API_KEY'), 'GOOGLE_API_KEY is required'
print('Gemini API key configured.')


## 3. Confirm Node / NPM (required by official MCP servers)

Many MCP servers are distributed as Node packages run via `npx`. Colab usually
ships with Node already installed; this cell verifies it and installs it if missing.


In [ ]:
import shutil, subprocess

def have(cmd):
    return shutil.which(cmd) is not None

if not (have('node') and have('npx')):
    print('Installing Node + npm...')
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'nodejs', 'npm'], check=True)

print('node:', subprocess.check_output(['node', '--version']).decode().strip())
print('npx :', subprocess.check_output(['npx', '--version']).decode().strip())


## 4. Prepare a working directory + tiny git repo

The filesystem and git MCP servers both need a directory to operate on. We create
a small workspace with a couple of sample text files and initialize git so the
git server has something meaningful to inspect.


In [ ]:
import os, subprocess, textwrap
from pathlib import Path

WORKDIR = Path('/content/workspace').resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

(WORKDIR / 'README.md').write_text(textwrap.dedent('''
    # Demo workspace
    This is a small repo used by the Gemini + MCP agent demo.
    It contains a few text files and a Python script.
''').strip(), encoding='utf-8')

(WORKDIR / 'notes.txt').write_text(textwrap.dedent('''
    Day 1: setup project and dependencies.
    Day 2: implement core data loading.
    Day 3: add evaluation pipeline.
    Day 4: write final report.
''').strip(), encoding='utf-8')

(WORKDIR / 'main.py').write_text(textwrap.dedent('''
    def greet(name: str) -> str:
        return f"Hello, {name}!"

    if __name__ == "__main__":
        print(greet("world"))
''').strip(), encoding='utf-8')

# Init git repo so the git server has history
def run(cmd, cwd=WORKDIR):
    subprocess.run(cmd, cwd=cwd, check=True, capture_output=True)

if not (WORKDIR / '.git').exists():
    run(['git', 'init', '-q'])
    run(['git', 'config', 'user.email', 'demo@local'])
    run(['git', 'config', 'user.name', 'demo'])
    run(['git', 'add', '.'])
    run(['git', 'commit', '-q', '-m', 'initial workspace'])
    # Add a second commit so the log/diff tools have something to diff
    (WORKDIR / 'notes.txt').write_text((WORKDIR / 'notes.txt').read_text() + '\nDay 5: present to stakeholders.', encoding='utf-8')
    run(['git', 'add', '.'])
    run(['git', 'commit', '-q', '-m', 'add day 5 milestone'])

print('Workspace:', WORKDIR)
print('Files:', sorted(p.name for p in WORKDIR.iterdir() if p.name != '.git'))
print('Git log:')
print(subprocess.check_output(['git', 'log', '--oneline'], cwd=WORKDIR).decode())


## 5. Build the custom MCP server (`workspace_ops`)

Three domain tools that the agent can compose with the official servers:
- `count_lines_by_extension(root)` — counts text lines per file extension.
- `summarize_file(content, max_sentences)` — returns a quick extractive summary.
- `changelog_from_diff(diff_text)` — converts a raw `git diff` into bullet points.


In [ ]:
from pathlib import Path
import textwrap

server_path = Path('/content/workspace_ops_server.py')
server_path.write_text(textwrap.dedent('''
    from fastmcp import FastMCP
    from pathlib import Path
    from typing import Dict, List
    import re

    mcp = FastMCP(name="workspace_ops")

    @mcp.tool
    def count_lines_by_extension(root: str) -> Dict[str, int]:
        """Count total lines per file extension under a directory."""
        counts: Dict[str, int] = {}
        for path in Path(root).rglob("*"):
            if not path.is_file() or any(part.startswith(".") for part in path.parts):
                continue
            ext = path.suffix or "<none>"
            try:
                lines = sum(1 for _ in path.open("r", encoding="utf-8", errors="ignore"))
            except Exception:
                continue
            counts[ext] = counts.get(ext, 0) + lines
        return counts

    @mcp.tool
    def summarize_file(content: str, max_sentences: int = 3) -> Dict[str, object]:
        """Quick extractive summary: returns the first N sentence-like fragments."""
        # naive sentence split on . ! ? or newline
        chunks = re.split(r"(?<=[.!?])\\s+|\\n+", content.strip())
        chunks = [c.strip() for c in chunks if c.strip()]
        summary = " ".join(chunks[:max_sentences])
        return {
            "summary": summary,
            "total_sentences": len(chunks),
            "used_sentences": min(max_sentences, len(chunks)),
        }

    @mcp.tool
    def changelog_from_diff(diff_text: str) -> List[str]:
        """Turn a raw git diff into a short list of changelog bullet points."""
        bullets: List[str] = []
        current_file = None
        for line in diff_text.splitlines():
            if line.startswith("diff --git"):
                # e.g. 'diff --git a/notes.txt b/notes.txt'
                parts = line.split()
                current_file = parts[-1].lstrip("b/") if parts else None
            elif line.startswith("+") and not line.startswith("+++"):
                added = line[1:].strip()
                if added and current_file:
                    bullets.append(f"{current_file}: added '{added[:80]}'")
            elif line.startswith("-") and not line.startswith("---"):
                removed = line[1:].strip()
                if removed and current_file:
                    bullets.append(f"{current_file}: removed '{removed[:80]}'")
        return bullets[:20]

    if __name__ == "__main__":
        mcp.run(transport="stdio")
'''), encoding='utf-8')

print('Wrote custom MCP server to:', server_path)


## 6. Connect to the three MCP servers

We register all three servers with `MultiServerMCPClient` and load their tools.
`tool_name_prefix=True` namespaces each tool by server (e.g.,
`filesystem__read_file`, `git__git_log`, `workspace_ops__summarize_file`).


In [ ]:
import asyncio
import nest_asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient

nest_asyncio.apply()  # so asyncio works inside Colab's running loop

mcp_connections = {
    'filesystem': {
        'transport': 'stdio',
        'command': 'npx',
        'args': ['-y', '@modelcontextprotocol/server-filesystem', str(WORKDIR)],
    },
    'git': {
        'transport': 'stdio',
        'command': 'python',
        'args': ['-m', 'mcp_server_git', '--repository', str(WORKDIR)],
    },
    'workspace_ops': {
        'transport': 'stdio',
        'command': 'python',
        'args': [str(server_path)],
    },
}

# Install the Python-side git MCP server (it is a pip package)
%pip install -qU mcp-server-git

client = MultiServerMCPClient(mcp_connections, tool_name_prefix=True)
tools = asyncio.get_event_loop().run_until_complete(client.get_tools())

print(f'Total tools loaded: {len(tools)}')
for t in tools:
    print(f'  - {t.name}')


## 7. Build the Gemini ReAct agent

We use `create_react_agent` from LangGraph. The LLM decides which tool to call
at every step based only on the user's prompt and the tool descriptions exposed
by the MCP servers — there is no hard-coded sequence.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(
    model='gemini-2.0-flash',
    temperature=0,
)

SYSTEM_PROMPT = (
    'You are a workspace assistant. You have access to three groups of tools: '
    'filesystem operations, git operations, and custom workspace_ops utilities. '
    'Plan multi-step actions when needed: read files, inspect git state, then '
    'use the custom tools to summarize content or build a changelog. '
    'Always cite which files or commits you used.'
)

agent = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)
print('Agent ready with', len(tools), 'MCP tools.')


## 8. Run example tasks

Three tasks of increasing complexity. The agent decides the sequence of tool calls.


In [ ]:
def ask(prompt: str):
    print('=' * 70)
    print('USER:', prompt)
    print('-' * 70)
    result = asyncio.get_event_loop().run_until_complete(
        agent.ainvoke({'messages': [{'role': 'user', 'content': prompt}]})
    )
    final = result['messages'][-1].content
    print('AGENT:', final)
    print()
    return result

# Task 1: pure filesystem composition
_ = ask('List every file in the workspace and tell me which one is the largest.')


In [ ]:
# Task 2: git + custom_ops composition
_ = ask(
    'Show me the last two git commits, then take the diff between them and '
    'turn it into a short changelog using the workspace_ops tool.'
)


In [ ]:
# Task 3: all three servers composed
_ = ask(
    'Read notes.txt, summarize it in 2 sentences using workspace_ops, then '
    'tell me how many total lines we have per file extension across the repo.'
)


## 9. Reflection

**What this demonstrates.** The agent uses tools provided by three separate MCP
servers and decides at runtime which to invoke, in which order, and how to feed
the output of one into the next (e.g., git diff → custom changelog). The control
flow is **tool-driven**, not hard-coded: switching to a different theme would
mostly be a matter of changing the system prompt and the custom server's tools.

**Why MCP matters here.** Each server is a standalone process that we don't have
to rewrite — the filesystem and git servers are official, off-the-shelf packages,
and our `workspace_ops` server is a thin FastMCP wrapper around three Python
functions. Adding a fourth server (e.g., a database connector) would be one line
in `mcp_connections`, and the agent would automatically gain those tools.

**Limitations and next steps.** Stdio MCP servers run as subprocesses in Colab,
which means they restart with the kernel; production deployments would use an
HTTP transport or a persistent process supervisor. Token usage also adds up when
the agent chains many tool calls — for production, cap iterations and cache
intermediate results.
